In [2]:
#Resize_1024
import cv2
import os
from pathlib import Path

# ------------ USER CONFIG -----------------
input_folder = Path("/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/CV_images/cv2_qu/img")
output_folder = Path("/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/CV_images/cv2_qu/resize_img")
target_size = (1024, 1024)  # width, height
# ------------------------------------------

output_folder.mkdir(parents=True, exist_ok=True)

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

for img_path in sorted(input_folder.iterdir()):
    if img_path.suffix.lower() not in image_exts:
        continue

    img = cv2.imread(str(img_path))
    if img is None:
        print(f"Failed to read: {img_path}")
        continue

    # Resize image
    resized = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)

    # Save in output folder
    out_path = output_folder / img_path.name
    cv2.imwrite(str(out_path), resized)

    print(f"Saved: {out_path}")

print("\nDone! All images resized to 1024×1024.")


Saved: /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/CV_images/cv2_qu/resize_img/10.9.8.jpg
Saved: /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/CV_images/cv2_qu/resize_img/1011.11.9.jpg
Saved: /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/CV_images/cv2_qu/resize_img/1022.12.6.jpg
Saved: /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/CV_images/cv2_qu/resize_img/1059.11.3.jpg
Saved: /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/CV_images/cv2_qu/resize_img/1124.11.6.jpg
Saved: /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/CV_images/cv2_qu/resize_img/1127.12.9.jpg
Saved: /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/CV_images/cv2_qu/resize_img/1128.12.3.jpg
Saved: /home/khushi/Pixonate/New_Anemia/R_new_anemia/model

In [10]:
import os
import random
import shutil
from pathlib import Path

# =========================
# CONFIG
# =========================
IMG_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/color_box_outputs/Data/CV_images"
LBL_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/color_box_outputs/Data/All/Otxt"
OUT_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/color_box_outputs/Data/All/color_split_all"

TRAIN_RATIO = 0.7
VAL_RATIO = 0.2
TEST_RATIO = 0.1

SEED = 42
IMG_EXTS = [".jpg", ".jpeg", ".png"]

# =========================
# SETUP
# =========================
random.seed(SEED)

for split in ["train", "val", "test"]:
    os.makedirs(f"{OUT_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{OUT_DIR}/labels/{split}", exist_ok=True)

# =========================
# COLLECT VALID FILES
# =========================
pairs = []

for img_name in os.listdir(IMG_DIR):
    if Path(img_name).suffix.lower() not in IMG_EXTS:
        continue

    lbl_path = os.path.join(LBL_DIR, Path(img_name).stem + ".txt")
    if not os.path.exists(lbl_path):
        continue

    pairs.append(img_name)

print(f"Total valid image-label pairs: {len(pairs)}")

# =========================
# SHUFFLE & SPLIT
# =========================
random.shuffle(pairs)

n_total = len(pairs)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)

train_files = pairs[:n_train]
val_files = pairs[n_train:n_train + n_val]
test_files = pairs[n_train + n_val:]

# =========================
# COPY FILES
# =========================
def copy_files(file_list, split):
    for img_name in file_list:
        shutil.copy(
            os.path.join(IMG_DIR, img_name),
            f"{OUT_DIR}/images/{split}/{img_name}"
        )

        shutil.copy(
            os.path.join(LBL_DIR, Path(img_name).stem + ".txt"),
            f"{OUT_DIR}/labels/{split}/{Path(img_name).stem}.txt"
        )

copy_files(train_files, "train")
copy_files(val_files, "val")
copy_files(test_files, "test")

# =========================
# SUMMARY
# =========================
print("✅ YOLO data split completed")
print(f"Train: {len(train_files)}")
print(f"Val  : {len(val_files)}")
print(f"Test : {len(test_files)}")


Total valid image-label pairs: 682
✅ YOLO data split completed
Train: 477
Val  : 136
Test : 69


In [ ]:
from ultralytics import YOLO

model = YOLO(
    "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/CV_images/cv2_qu/eye_split/best1.pt"
)

model.train(
    data="/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/eye_outputs/Finetune_data/data.yaml",
    epochs=100,
    imgsz=1024,
    batch=8,
    device=0
)


In [1]:
# Inference
from ultralytics import YOLO
import os

# =========================
# CONFIG
# =========================
MODEL_PATH = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/color_box_outputs/Data/All/color_split_all/F_Color_last.pt"
TEST_IMG_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/color_box_outputs/Data/All/color_split_all/images/test"
OUTPUT_DIR = "/home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/color_box_outputs/Data/All/color_split_all/Prediction"

IMG_SIZE = 1024
CONF_THRES = 0.75

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# LOAD MODEL
# =========================
model = YOLO(MODEL_PATH)

# =========================
# RUN INFERENCE
# =========================
results = model.predict(
    source=TEST_IMG_DIR,
    imgsz=IMG_SIZE,
    conf=CONF_THRES,
    save=True,
    project=OUTPUT_DIR,
    name="pred_images",
    device="cpu"
)

print("✅ Inference completed.")



image 1/69 /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/color_box_outputs/Data/All/color_split_all/images/test/1044.11.0.jpg: 1024x768 24 boxs, 316.5ms
image 2/69 /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/color_box_outputs/Data/All/color_split_all/images/test/1048.10.3.jpg: 1024x768 24 boxs, 271.0ms
image 3/69 /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/color_box_outputs/Data/All/color_split_all/images/test/1053.11.5.jpg: 1024x768 24 boxs, 256.3ms
image 4/69 /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/color_box_outputs/Data/All/color_split_all/images/test/1054.12.1.jpg: 1024x768 24 boxs, 249.7ms
image 5/69 /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/color_box_outputs/Data/All/color_split_all/images/test/1059.11.3.jpg: 1024x768 25 boxs, 244.4ms
image 6/69 /home/khushi/Pixonate/New_Anemia/R_new_anemia/model_outputs/color_box_outputs/Data/All/color_split_all/images/test/1072.9.4.jpg: 1024x768 24 boxs, 336.9m